# Regressão de Potência — RandomizedSearchCV
**Projeto Final — CKP8277 Aprendizagem Automática · UFC**

**Aluno:** Diego Melo — 603127

Modelos avaliados via RandomizedSearchCV (100 iterações, 5-fold CV):
- Random Forest Regressor
- XGBoost Regressor
- LightGBM Regressor
- CatBoost Regressor
- KNN Regressor
- SARIMA Regression (baseline temporal)

In [10]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


---
## 0. Instalação de Dependências

Execute a célula abaixo **apenas uma vez** para instalar todas as bibliotecas necessárias.

In [11]:
%pip install -q \
    numpy \
    pandas \
    matplotlib \
    seaborn \
    scipy \
    scikit-learn \
    xgboost \
    lightgbm \
    catboost \
    pmdarima \
    joblib \
    "mlflow>=2.14,<3" \
    "protobuf<5" \
    openpyxl \
    statsmodels

---
## 1. Configuração Global

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import json
import joblib
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

# Sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Boosting
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# SARIMA
from pmdarima import auto_arima

# MLflow
import mlflow
import mlflow.sklearn

# ── Detecção de GPU ───────────────────────────────────────────────────────────
def detect_gpu():
    try:
        import subprocess
        result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
        if result.returncode == 0: return True
    except: pass
    return False

GPU_AVAILABLE = detect_gpu()
XGB_DEVICE    = 'cuda' if GPU_AVAILABLE else 'cpu'
LGBM_DEVICE   = 'gpu'  if GPU_AVAILABLE else 'cpu'
CAT_TASK_TYPE = 'GPU'  if GPU_AVAILABLE else 'CPU'

# ── Tema visual ───────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='Set2')
DPI, FIG_W, FIG_H = 120, 14, 4.5
RANDOM_STATE, N_ITER, CV_FOLDS = 42, 100, 5
SCORING = 'neg_root_mean_squared_error'

# ── Diretórios de artefatos (Apontando para o Drive) ──────────────────────────
BASE_DRIVE_PATH = '/content/drive/MyDrive/Mestrado/Aprendizagem Automática/projeto final/experiment'
MODELS_DIR  = os.path.join(BASE_DRIVE_PATH, 'models/')
FIGURES_DIR = os.path.join(BASE_DRIVE_PATH, 'figures/')
RESULTS_DIR = os.path.join(BASE_DRIVE_PATH, 'results/')
MLFLOW_DIR  = os.path.join(BASE_DRIVE_PATH, 'mlruns/')

os.makedirs(MODELS_DIR,  exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f'Artefatos serão salvos em: {BASE_DRIVE_PATH}')

---
## 2. Configuração do MLflow

In [13]:
EXPERIMENT_NAME = 'regressao-potencia-fotovoltaica'

# Configura o MLflow para salvar no Drive
mlflow.set_tracking_uri(f'file://{MLFLOW_DIR}')
mlflow.set_experiment(EXPERIMENT_NAME)

print(f'Experimento MLflow configurado no Drive: {MLFLOW_DIR}')

Experimento MLflow configurado no Drive: /content/drive/MyDrive/Mestrado/Aprendizagem Automática/projeto final/experiment/mlruns/


---
## 3. Carregamento e Pré-processamento

In [14]:
DATA_PATH = '/content/drive/MyDrive/Mestrado/Aprendizagem Automática/projeto final/articles/vagner_proposal/dataresult_en.csv'

df = pd.read_csv(DATA_PATH, parse_dates=['DATETIME'])
df = df.sort_values('DATETIME').reset_index(drop=True)

TARGET   = 'POWERFROMPANELS'
FEATURES = ['IRRADIATION', 'ENVTEMPSTATION', 'WINDSPEED', 'ENVTEMPPANELS']

print(f'Shape original: {df.shape}')
print(f'Nulos: {df.isnull().sum().sum()}')

Shape original: (35336, 6)
Nulos: 0


In [15]:
# ── Engenharia de features temporais ─────────────────────────────────────────
df['HOUR']       = df['DATETIME'].dt.hour
df['MINUTE']     = df['DATETIME'].dt.minute
df['MONTH']      = df['DATETIME'].dt.month
df['DOY']        = df['DATETIME'].dt.dayofyear   # dia do ano (sazonalidade)
df['HOUR_SIN']   = np.sin(2 * np.pi * df['HOUR'] / 24)
df['HOUR_COS']   = np.cos(2 * np.pi * df['HOUR'] / 24)
df['MONTH_SIN']  = np.sin(2 * np.pi * df['MONTH'] / 12)
df['MONTH_COS']  = np.cos(2 * np.pi * df['MONTH'] / 12)

# Feature de interação física: irradiância × temperatura (aquecimento)
df['IRRAD_TEMP'] = df['IRRADIATION'] * df['ENVTEMPPANELS']

ALL_FEATURES = FEATURES + [
    'HOUR', 'MINUTE', 'MONTH', 'DOY',
    'HOUR_SIN', 'HOUR_COS', 'MONTH_SIN', 'MONTH_COS',
    'IRRAD_TEMP'
]

print(f'Features totais: {len(ALL_FEATURES)}')
print(ALL_FEATURES)

Features totais: 13
['IRRADIATION', 'ENVTEMPSTATION', 'WINDSPEED', 'ENVTEMPPANELS', 'HOUR', 'MINUTE', 'MONTH', 'DOY', 'HOUR_SIN', 'HOUR_COS', 'MONTH_SIN', 'MONTH_COS', 'IRRAD_TEMP']


In [16]:
# ── Split temporal 80/20 ANTES de qualquer limpeza ───────────────────────────
# Importante: o split deve ocorrer ANTES da remoção de outliers para evitar
# que estatísticas do conjunto de teste influenciem a limpeza do treino.
split_idx = int(len(df) * 0.8)

df_train_raw = df.iloc[:split_idx].copy()
df_test_raw  = df.iloc[split_idx:].copy()

# ── Remoção de outliers extremos apenas no treino (Z-score > 4) ──────────────
# Z-score calculado SOMENTE sobre o treino — evita data leakage do teste.
z_train = np.abs(stats.zscore(df_train_raw[TARGET]))
df_train = df_train_raw[z_train < 4].copy().reset_index(drop=True)
df_test  = df_test_raw.copy().reset_index(drop=True)

print(f'Treino original : {len(df_train_raw):,} | após remover outliers: {len(df_train):,} '
      f'({len(df_train_raw) - len(df_train)} removidos)')
print(f'Teste           : {len(df_test):,} (sem remoção de outliers)')
print(f'Período treino  : {df_train["DATETIME"].iloc[0].date()} → {df_train["DATETIME"].iloc[-1].date()}')
print(f'Período teste   : {df_test["DATETIME"].iloc[0].date()} → {df_test["DATETIME"].iloc[-1].date()}')

X_train = df_train[ALL_FEATURES].values
y_train = df_train[TARGET].values
X_test  = df_test[ALL_FEATURES].values
y_test  = df_test[TARGET].values

# ── Normalização com RobustScaler ────────────────────────────────────────────
# fit() apenas no treino — transform() aplicado separadamente no teste.
scaler     = RobustScaler()
X_train_sc = scaler.fit_transform(X_train)   # aprende mediana/IQR só do treino
X_test_sc  = scaler.transform(X_test)        # aplica os mesmos parâmetros ao teste

# TimeSeriesSplit para o CV (respeita a ordem temporal dentro do treino)
tscv = TimeSeriesSplit(n_splits=CV_FOLDS)
print('Scaler ajustado. TimeSeriesSplit configurado.')

Treino original : 28,268 | após remover outliers: 28,268 (0 removidos)
Teste           : 7,068 (sem remoção de outliers)
Período treino  : 2021-05-01 → 2021-10-04
Período teste   : 2021-10-04 → 2021-10-13
Scaler ajustado. TimeSeriesSplit configurado.


---
## 4. Funções Auxiliares

In [ ]:
import sys

PALETTE = 'Set2'
PRIMARY = '#2D7DD2'
ACCENT  = '#F18F01'
NEUTRAL = '#6C757D'

def log(msg):
    """Print com flush imediato — garante visibilidade em Colab/Kaggle."""
    print(msg, flush=True)
    sys.stdout.flush()


def evaluate(name, model, X_te, y_te):
    """Avalia um modelo já treinado no conjunto de teste. Não re-treina."""
    y_pred = np.clip(model.predict(X_te), 0, None)
    return {
        'Modelo' : name,
        'RMSE'   : np.sqrt(mean_squared_error(y_te, y_pred)),
        'MAE'    : mean_absolute_error(y_te, y_pred),
        'R²'     : r2_score(y_te, y_pred),
        'MAPE%'  : np.mean(np.abs((y_te - y_pred) / (y_te + 1e-6))) * 100,
        'y_pred' : y_pred,
    }


def run_random_search(name, estimator, param_dist, X_tr, y_tr):
    safe_name  = name.replace(' ', '_')
    cache_path = os.path.join(MODELS_DIR, f'{safe_name}.joblib')

    log(f'\n{"="*55}')
    log(f'  Modelo : {name}')
    log(f'  Busca  : {N_ITER} iterações × {CV_FOLDS}-fold TimeSeriesSplit')
    log(f'{"="*55}')

    rs = RandomizedSearchCV(
        estimator, param_dist,
        n_iter=N_ITER, cv=tscv,
        scoring=SCORING,
        random_state=RANDOM_STATE,
        n_jobs=-1 if not GPU_AVAILABLE else 1,
        verbose=0,    # desligado — usamos log() manual abaixo
        refit=True,
    )
    rs.fit(X_tr, y_tr)

    best_model = rs.best_estimator_
    cv_rmse    = -rs.best_score_

    # Avalia no teste — sem re-treino
    res = evaluate(name, best_model, X_test_sc, y_test)

    # Relatório por modelo logo após terminar
    log(f'\n  ✔ {name} concluído')
    log(f'  ┌─────────────────────────────────────')
    log(f'  │  CV RMSE  : {cv_rmse:>10,.2f} W')
    log(f'  │  RMSE     : {res["RMSE"]:>10,.2f} W')
    log(f'  │  MAE      : {res["MAE"]:>10,.2f} W')
    log(f'  │  R²       : {res["R²"]:>10.4f}')
    log(f'  │  MAPE     : {res["MAPE%"]:>10.2f} %')
    log(f'  └─────────────────────────────────────')
    log(f'  Melhores parâmetros:')
    for k, v in rs.best_params_.items():
        log(f'    {k}: {v}')

    # Top 10 configurações
    cv_df  = pd.DataFrame(rs.cv_results_)
    top10  = cv_df.sort_values('mean_test_score', ascending=False).head(10).copy()
    top10['RMSE_CV'] = -top10['mean_test_score']
    pcols  = [c for c in top10.columns if c.startswith('param_')]
    top10_path = os.path.join(RESULTS_DIR, f'top10_{safe_name}.csv')
    top10[pcols + ['RMSE_CV', 'mean_fit_time']].to_csv(top10_path, index=False)

    # Salva modelo
    joblib.dump(best_model, cache_path)

    # MLflow
    with mlflow.start_run(run_name=name):
        mlflow.log_params({k: str(v) for k, v in rs.best_params_.items()})
        mlflow.log_metrics({
            'cv_rmse' : cv_rmse,
            'rmse'    : res['RMSE'],
            'mae'     : res['MAE'],
            'r2'      : res['R²'],
            'mape_pct': res['MAPE%'],
        })
        mlflow.sklearn.log_model(best_model, 'model')
        mlflow.log_artifact(top10_path, 'search_results')
        scaler_path = os.path.join(MODELS_DIR, 'scaler.joblib')
        joblib.dump(scaler, scaler_path)
        mlflow.log_artifact(scaler_path, 'preprocessing')

    return best_model, rs.best_params_, cv_rmse, res


results     = []
best_models = {}
log('Funções definidas.')

---
## 5. Random Forest Regressor

In [ ]:
rf_params = {
    'n_estimators'    : [100, 200, 300, 500, 700],
    'max_depth'       : [None, 8, 10, 12, 15, 20],
    'min_samples_leaf': [1, 2, 3, 4],
    'min_samples_split': [2, 4, 6, 8],
    'max_features'    : ['sqrt', 'log2', 3, 4, 5],
    'bootstrap'       : [True, False],
    'max_samples'     : [0.7, 0.8, 0.9, None],
}

rf_best, rf_best_params, rf_cv_rmse, rf_res = run_random_search(
    'Random Forest',
    RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    rf_params, X_train_sc, y_train
)
rf_res['CV_RMSE'] = rf_cv_rmse
results.append(rf_res)
best_models['Random Forest'] = rf_best
print(f'  Teste → RMSE={rf_res["RMSE"]:,.1f}  MAE={rf_res["MAE"]:,.1f}  R²={rf_res["R²"]:.4f}')

---
## 6. XGBoost Regressor

In [ ]:
xgb_params = {
    'n_estimators'    : [100, 200, 500, 800, 1000],
    'learning_rate'   : [0.01, 0.03, 0.05, 0.1, 0.15],
    'max_depth'       : [4, 5, 6, 7, 8],
    'subsample'       : [0.7, 0.75, 0.8, 0.85, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5, 7, 10],
    'gamma'           : [0, 0.1, 0.2, 0.5],
    'reg_alpha'       : [0, 0.01, 0.1, 0.5],
    'reg_lambda'      : [0.5, 1.0, 2.0, 5.0],
}

xgb_best, xgb_best_params, xgb_cv_rmse, xgb_res = run_random_search(
    'XGBoost',
    XGBRegressor(
        random_state=RANDOM_STATE,
        device=XGB_DEVICE,
        n_jobs=-1 if not GPU_AVAILABLE else 1,
        verbosity=0,
    ),
    xgb_params, X_train_sc, y_train
)
xgb_res['CV_RMSE'] = xgb_cv_rmse
results.append(xgb_res)
best_models['XGBoost'] = xgb_best
print(f'  Teste → RMSE={xgb_res["RMSE"]:,.1f}  MAE={xgb_res["MAE"]:,.1f}  R²={xgb_res["R²"]:.4f}')

---
## 7. LightGBM Regressor

In [ ]:
lgbm_params = {
    'n_estimators'     : [200, 500, 700, 1000],
    'learning_rate'    : [0.01, 0.03, 0.05, 0.1],
    'num_leaves'       : [31, 50, 63, 80, 100],
    'max_depth'        : [-1, 5, 6, 7, 8, 10],
    'min_child_samples': [10, 20, 30, 50, 100],
    'subsample'        : [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree' : [0.7, 0.8, 0.9, 1.0],
    'reg_alpha'        : [0, 0.01, 0.1, 0.5],
    'reg_lambda'       : [0, 0.01, 0.1, 1.0],
}

lgbm_best, lgbm_best_params, lgbm_cv_rmse, lgbm_res = run_random_search(
    'LightGBM',
    LGBMRegressor(
        random_state=RANDOM_STATE,
        device=LGBM_DEVICE,
        n_jobs=-1 if not GPU_AVAILABLE else 1,
        verbose=-1,
    ),
    lgbm_params, X_train_sc, y_train
)
lgbm_res['CV_RMSE'] = lgbm_cv_rmse
results.append(lgbm_res)
best_models['LightGBM'] = lgbm_best
print(f'  Teste → RMSE={lgbm_res["RMSE"]:,.1f}  MAE={lgbm_res["MAE"]:,.1f}  R²={lgbm_res["R²"]:.4f}')

---
## 8. CatBoost Regressor

In [ ]:
cat_params = {
    'iterations'        : [200, 500, 700, 1000],
    'learning_rate'     : [0.01, 0.03, 0.05, 0.1],
    'depth'             : [4, 5, 6, 7, 8, 10],
    'l2_leaf_reg'       : [1, 2, 3, 5, 7],
    'bagging_temperature': [0, 0.5, 1.0, 2.0],
    'random_strength'   : [0, 0.5, 1.0, 2.0],
    'border_count'      : [32, 64, 128],
}

cat_best, cat_best_params, cat_cv_rmse, cat_res = run_random_search(
    'CatBoost',
    CatBoostRegressor(
        random_state=RANDOM_STATE,
        task_type=CAT_TASK_TYPE,
        verbose=0,
    ),
    cat_params, X_train_sc, y_train
)
cat_res['CV_RMSE'] = cat_cv_rmse
results.append(cat_res)
best_models['CatBoost'] = cat_best
print(f'  Teste → RMSE={cat_res["RMSE"]:,.1f}  MAE={cat_res["MAE"]:,.1f}  R²={cat_res["R²"]:.4f}')

---
## 9. KNN Regressor

In [ ]:
knn_params = {
    'n_neighbors': list(range(3, 26)),
    'weights'    : ['uniform', 'distance'],
    'metric'     : ['euclidean', 'manhattan', 'minkowski'],
    'p'          : [1, 2],
    'leaf_size'  : [20, 30, 40],
    'algorithm'  : ['auto', 'ball_tree', 'kd_tree'],
}

knn_best, knn_best_params, knn_cv_rmse, knn_res = run_random_search(
    'KNN',
    KNeighborsRegressor(n_jobs=-1),
    knn_params, X_train_sc, y_train
)
knn_res['CV_RMSE'] = knn_cv_rmse
results.append(knn_res)
best_models['KNN'] = knn_best
print(f'  Teste → RMSE={knn_res["RMSE"]:,.1f}  MAE={knn_res["MAE"]:,.1f}  R²={knn_res["R²"]:.4f}')

---
## 10. SARIMA Regression

> SARIMA é um modelo de série temporal puro — não usa features externas na versão base.  
> Avaliamos com `auto_arima` (pmdarima) em série diária agregada para viabilizar a estimação.  
> Serve como **baseline temporal** na comparação com os modelos supervisionados.

In [ ]:
# Agregação diária — usa df_train (série limpa) + df_test para manter split correto
df_daily_train = (
    df_train.set_index('DATETIME')[TARGET]
    .resample('D').sum()
    .dropna()
)
df_daily_test = (
    df_test.set_index('DATETIME')[TARGET]
    .resample('D').sum()
    .dropna()
)
df_daily = pd.concat([df_daily_train, df_daily_test])

train_s = df_daily_train
test_s  = df_daily_test

print(f'Série diária — treino: {len(train_s)} dias | teste: {len(test_s)} dias')

sarima_model = auto_arima(
    train_s,
    seasonal=True, m=7,
    start_p=0, max_p=3,
    start_q=0, max_q=3,
    start_P=0, max_P=2,
    start_Q=0, max_Q=2,
    d=None, D=None,
    information_criterion='aic',
    stepwise=True,
    error_action='ignore',
    suppress_warnings=True,
    n_fits=50
)
print(sarima_model.summary())

In [ ]:
sarima_pred = sarima_model.predict(n_periods=len(test_s))
sarima_pred = np.clip(sarima_pred, 0, None)

rmse_s = np.sqrt(mean_squared_error(test_s, sarima_pred))
mae_s  = mean_absolute_error(test_s, sarima_pred)
r2_s   = r2_score(test_s, sarima_pred)
mape_s = np.mean(np.abs((test_s.values - sarima_pred) / (test_s.values + 1e-6))) * 100

print(f'SARIMA → RMSE={rmse_s:,.1f}  MAE={mae_s:,.1f}  R²={r2_s:.4f}  MAPE%={mape_s:.2f}')
print('(Nota: métricas em Wh diário — escala diferente dos modelos por minuto)')

# Registra SARIMA no MLflow
with mlflow.start_run(run_name='SARIMA'):
    mlflow.set_tags({
        'model'      : 'SARIMA',
        'cv_strategy': 'auto_arima stepwise',
        'dataset'    : 'estacao_meteorologica.csv (agregado diário)',
        'granularity': 'daily',
    })
    mlflow.log_params({
        'order'         : str(sarima_model.order),
        'seasonal_order': str(sarima_model.seasonal_order),
        'm'             : 7,
    })
    mlflow.log_metrics({'rmse': rmse_s, 'mae': mae_s, 'r2': r2_s, 'mape_pct': mape_s})

    sarima_path = os.path.join(MODELS_DIR, 'SARIMA.joblib')
    joblib.dump(sarima_model, sarima_path)
    mlflow.log_artifact(sarima_path, artifact_path='model')
    print(f'  [SARIMA] MLflow run_id: {mlflow.active_run().info.run_id}')

sarima_res = {
    'Modelo' : 'SARIMA', 'RMSE': rmse_s, 'MAE': mae_s,
    'R²'     : r2_s,     'MAPE%': mape_s, 'CV_RMSE': float('nan'),
    'y_pred' : sarima_pred,
}
results.append(sarima_res)

---
## 11. Comparação de Resultados

In [ ]:
# Tabela resumo (sem SARIMA na comparação direta por escala diferente)
res_df = pd.DataFrame([
    {k: v for k, v in r.items() if k != 'y_pred'}
    for r in results
]).set_index('Modelo')

res_df_sorted = res_df.sort_values('RMSE')
print(res_df_sorted.round(2).to_string())

In [ ]:
# Gráfico comparativo — modelos supervisionados (excluindo SARIMA)
supervised = res_df.drop('SARIMA', errors='ignore')
metrics    = ['RMSE', 'MAE', 'R²', 'MAPE%']
pal = sns.color_palette(PALETTE, len(supervised))

fig, axes = plt.subplots(1, 4, figsize=(FIG_W * 1.1, FIG_H), dpi=DPI)
fig.suptitle('Comparação dos Modelos — Conjunto de Teste')

for ax, metric in zip(axes, metrics):
    data = supervised[metric].sort_values(ascending=(metric != 'R²'))
    bars = ax.barh(data.index, data.values,
                   color=pal[:len(data)], edgecolor='white', linewidth=0.5)
    ax.set_title(metric)
    ax.set_xlabel(metric)
    for bar, val in zip(bars, data.values):
        ax.text(val * 1.01, bar.get_y() + bar.get_height()/2,
                f'{val:.2f}', va='center', fontsize=8)

plt.tight_layout()
plt.show()

---
## 12. Predito vs Real — Melhor Modelo

In [ ]:
# Identifica melhor modelo supervisionado (menor RMSE teste)
best_name = supervised['RMSE'].idxmin()
best_pred = next(r['y_pred'] for r in results if r['Modelo'] == best_name)

print(f'Melhor modelo: {best_name}')

fig, axes = plt.subplots(1, 2, figsize=(FIG_W, FIG_H + 0.5), dpi=DPI)
fig.suptitle(f'{best_name} — Predito vs Real (conjunto de teste)')

# Scatter predito vs real
ax = axes[0]
ax.scatter(y_test, best_pred, alpha=0.15, s=6, color=PRIMARY, linewidths=0)
lim = max(y_test.max(), best_pred.max())
ax.plot([0, lim], [0, lim], color=ACCENT, linewidth=1.5, linestyle='--', label='Ideal')
ax.set_xlabel('Real (W)'); ax.set_ylabel('Predito (W)')
ax.set_title('Scatter: Real vs Predito'); ax.legend()

# Série temporal — primeiros 500 pontos do teste
ax = axes[1]
n_show = 500
idx = np.arange(n_show)
ax.plot(idx, y_test[:n_show], color=NEUTRAL, linewidth=0.8, label='Real', alpha=0.8)
ax.plot(idx, best_pred[:n_show], color=ACCENT, linewidth=0.8, label='Predito', alpha=0.9)
ax.set_xlabel('Amostra'); ax.set_ylabel('Potência (W)')
ax.set_title(f'Série Temporal (primeiros {n_show} pontos do teste)')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Distribuição dos resíduos — melhor modelo
residuals = y_test - best_pred

fig, axes = plt.subplots(1, 3, figsize=(FIG_W, FIG_H), dpi=DPI)
fig.suptitle(f'{best_name} — Análise de Resíduos')

ax = axes[0]
sns.histplot(residuals, bins=60, kde=True, color=PRIMARY, ax=ax)
ax.axvline(0, color=ACCENT, linestyle='--', linewidth=1.5)
ax.set_title('Distribuição dos Resíduos'); ax.set_xlabel('Resíduo (W)')

ax = axes[1]
ax.scatter(best_pred, residuals, alpha=0.15, s=5, color=PRIMARY, linewidths=0)
ax.axhline(0, color=ACCENT, linestyle='--', linewidth=1.5)
ax.set_xlabel('Predito (W)'); ax.set_ylabel('Resíduo (W)')
ax.set_title('Resíduo vs Predito')

ax = axes[2]
stats.probplot(residuals, dist='norm', plot=ax)
ax.set_title('Q-Q Plot dos Resíduos')

plt.tight_layout()
plt.show()

---
## 13. Importância de Features — Melhor Modelo

In [ ]:
best_model_obj = best_models[best_name]
has_importance  = hasattr(best_model_obj, 'feature_importances_')

if has_importance:
    imp = pd.Series(best_model_obj.feature_importances_, index=ALL_FEATURES)
    imp = imp.sort_values(ascending=True)

    fig, ax = plt.subplots(figsize=(9, 5), dpi=DPI)
    colors = [PRIMARY if v == imp.max() else NEUTRAL for v in imp.values]
    ax.barh(imp.index, imp.values, color=colors, edgecolor='white')
    ax.set_title(f'Importância de Features — {best_name}')
    ax.set_xlabel('Importância')
    plt.tight_layout()
    plt.show()
else:
    print(f'{best_name} não expõe feature_importances_. Usando permutation importance...')
    from sklearn.inspection import permutation_importance
    perm = permutation_importance(best_model_obj, X_test_sc, y_test,
                                   n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
    imp = pd.Series(perm.importances_mean, index=ALL_FEATURES).sort_values(ascending=True)

    fig, ax = plt.subplots(figsize=(9, 5), dpi=DPI)
    ax.barh(imp.index, imp.values, color=PRIMARY, edgecolor='white')
    ax.set_title(f'Permutation Importance — {best_name}')
    ax.set_xlabel('Redução média no RMSE')
    plt.tight_layout()
    plt.show()

---
## 14. SARIMA — Visualização da Predição Diária

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(FIG_W, FIG_H * 2), dpi=DPI)
fig.suptitle('SARIMA — Geração Diária Total (Wh)')

ax = axes[0]
ax.plot(df_daily.index, df_daily.values, color=NEUTRAL, linewidth=1, label='Real')
ax.plot(test_s.index, sarima_pred, color=ACCENT, linewidth=1.5, label='SARIMA Predito')
ax.axvline(test_s.index[0], color='red', linestyle='--', linewidth=1, alpha=0.7, label='Início teste')
ax.set_title('Série completa + Predição'); ax.set_ylabel('Wh/dia'); ax.legend()

ax = axes[1]
ax.plot(test_s.index, test_s.values, color=NEUTRAL, linewidth=1.2, label='Real')
ax.plot(test_s.index, sarima_pred, color=ACCENT, linewidth=1.5, label='SARIMA Predito')
ax.fill_between(test_s.index, test_s.values, sarima_pred,
                alpha=0.2, color='red', label='Erro')
ax.set_title(f'Conjunto de Teste  |  RMSE={rmse_s:,.0f} Wh  R²={r2_s:.3f}')
ax.set_ylabel('Wh/dia'); ax.legend()

plt.tight_layout()
plt.show()

---
## 15. Melhores Hiperparâmetros Encontrados

In [ ]:
best_params_all = {
    'Random Forest': rf_best_params,
    'XGBoost'      : xgb_best_params,
    'LightGBM'     : lgbm_best_params,
    'CatBoost'     : cat_best_params,
    'KNN'          : knn_best_params,
    'SARIMA'       : {'order': sarima_model.order, 'seasonal_order': sarima_model.seasonal_order},
}

print('=' * 60)
print('MELHORES HIPERPARÂMETROS — RandomizedSearchCV')
print('=' * 60)
for name, params in best_params_all.items():
    print(f'\n[{name}]')
    for k, v in params.items():
        print(f'  {k}: {v}')

---
## 16. Resumo Final

In [ ]:
print('=' * 60)
print('RESUMO FINAL — Regressão de Potência')
print('=' * 60)
print(f'\nDataset treino : {len(df_train):,} registros | Features: {len(ALL_FEATURES)}')
print(f'Dataset teste  : {len(df_test):,} registros')
print(f'Treino : {X_train.shape[0]:,} | Teste: {X_test.shape[0]:,}')
print(f'CV     : TimeSeriesSplit ({CV_FOLDS} folds) | N_iter: {N_ITER}')
print(f'Scaler : RobustScaler')
print()
print('Ranking por RMSE (conjunto de teste):')
print(res_df_sorted[['RMSE','MAE','R²','MAPE%']].round(2).to_string())
print(f'\nMelhor modelo: {best_name}')